In [1]:
from pyspark.sql.functions import when, col, udf
from pyspark.sql.types import StringType
import reverse_geocoder as rg
from delta.tables import DeltaTable
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

StatementMeta(, aedbe75c-b0e9-4cb5-9113-2d76d736d333, 5, Finished, Available, Finished, False)

In [ ]:
print("========== GOLD PARAMETERS ==========")
print(f"start_date = '{start_date}'")
print(f"end_date   = '{end_date}'")
print("====================================")

In [3]:
df= ( \
    spark.read.table("silver_events")
    
    .filter(
        (col('time')>=start_date) &
        (col('time')< end_date)
    )
)

StatementMeta(, aedbe75c-b0e9-4cb5-9113-2d76d736d333, 7, Finished, Available, Finished, False)

In [4]:
from pyspark.sql.functions import when, col

df_with_location_sig_class= (
    df
    .withColumn(
        'sig_class',
        when(df["sig"]< 100, "Low")
        .when(
            (df["sig"]>= 100) &
            (df["sig"]<500),
            "Moderate"
        )
        .otherwise("High")
    )
)

StatementMeta(, aedbe75c-b0e9-4cb5-9113-2d76d736d333, 8, Finished, Available, Finished, False)

In [5]:
def get_country_code(latitude, longitude):
    result = rg.search((latitude, longitude), mode=1)
    return result[0]["cc"]

country_code_udf = udf(get_country_code, StringType())

df_with_location_sig_class = (
    df_with_location_sig_class
    .withColumn(
        "country_code",
        country_code_udf(
            col("latitude"),
            col("longitude")
        )
    )
)

StatementMeta(, aedbe75c-b0e9-4cb5-9113-2d76d736d333, 9, Finished, Available, Finished, False)

In [ ]:
'''df_with_location_sig_class.write.format("delta").mode("overwrite").saveAsTable(
    "gold_events"
)'''

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [6]:
target=DeltaTable.forName(
    spark,
    "gold_events"
)

(
    target.alias("target")
    .merge(
        df_with_location_sig_class.alias("source"),
        "target.earthquake_id=source.earthquake_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

StatementMeta(, aedbe75c-b0e9-4cb5-9113-2d76d736d333, 10, Finished, Available, Finished, False)

In [ ]:
print("========== GOLD PROCESSING ==========")
print(f"Records selected from Silver: {df.count()}")
print("====================================")